# 07 — Export, demo ve README görselleri (Faz 7)

**Çalışma zamanı: T4 GPU yeterli** (export ve karşılaştırma için; demo CPU'da ONNX ile koşar). Toplam ~20–25 dk.

1. **ONNX export:** final model (E05) PyTorch'suz çalışan tek bir dosyaya çevrilir.
2. **Doğrulama:** ONNX + `predict.py` (sadece ONNX Runtime) ile PyTorch modelinin tahminleri karşılaştırılır; CPU hızı ölçülür.
3. **Demo:** Heinola test videolarının kareleri sırayla `predict.py`'den geçer; tespitler ve **zaman filtresi** (son N karenin K'sında görülürse alarm) videoya işlenir. Heinola, modelin hiç görmediği yangın.
4. **Olay bazında ölçüm:** ham kare tespiti ile filtreli alarm karşılaştırılır (yakalama, yanlış alarm, ilk alarm gecikmesi).
5. **README görselleri:** deney grafiği, hata grid'leri, demo GIF'i → `export/assets/`.

Not: Heinola kareleri videodan **örneklenmiş**; ardışık iki kare arasında gerçek zamanda birkaç saniye olabilir. Demo FPS'i bu yüzden temsilîdir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L
!pip -q install ultralytics==8.4.183 onnx onnxslim onnxruntime

In [ ]:
import os, re, sys, json, shutil, subprocess, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import yaml
from ultralytics import YOLO

ROOT   = Path('/content/drive/MyDrive/firesmoke')
REP    = ROOT / 'reports'
RUNS   = ROOT / 'runs'
CODE   = ROOT / 'code'               # predict.py burada
EXPORT = ROOT / 'export'
ASSETS = EXPORT / 'assets'

# --- Ayarlar -------------------------------------------------------------------
FINAL      = 'E05_y11s_640_dfire'
DSET       = 'firesmoke_v2'
K, N       = 3, 5          # alarm: son N karenin en az K'sında tespit
FPS        = 4             # demo videosu
MAX_VIDEOS = 4             # demoya girecek en uzun Heinola videosu sayısı
SHOW_GT    = True          # etiket kutularını ince yeşil çiz
WIDTH      = 960           # demo videosu genişliği
# ------------------------------------------------------------------------------

DS_ZIP = ROOT / 'datasets' / f'{DSET}.zip'
DATA   = Path('/content') / DSET
ASSETS.mkdir(parents=True, exist_ok=True)
IMGSZ  = int(yaml.safe_load((RUNS / FINAL / 'args.yaml').read_text())['imgsz'])
fe     = json.loads((REP / 'final_eval' / 'final_eval.json').read_text())
assert fe['final'] == FINAL, (fe['final'], FINAL)
THR    = {0: float(fe['thr']['fire']), 1: float(fe['thr']['smoke'])}
sys.path.insert(0, str(CODE))
from predict import Detector, AlarmFilter, draw
print('eşikler (06\'da val\'den seçilen):', THR, '· imgsz', IMGSZ)

## 1. Dataset'i aç ve ONNX'e çevir

In [ ]:
t0 = time.time()
if not (DATA / 'data.yaml').exists():
    subprocess.run(['unzip', '-q', '-o', str(DS_ZIP), '-d', str(DATA.parent)], check=True)
man = pd.read_csv(DATA / 'manifest.csv')
test = man[man.split == 'test'].reset_index(drop=True)
print(f'test {len(test)} ({time.time() - t0:.0f} sn)')

best = RUNS / FINAL / 'weights' / 'best.pt'
onnx_tmp = YOLO(str(best)).export(format='onnx', imgsz=IMGSZ, simplify=True, dynamic=False)
ONNX = EXPORT / f'firesmoke_{FINAL.split("_")[1]}_{IMGSZ}.onnx'
shutil.copy(onnx_tmp, ONNX)
shutil.copy(CODE / 'predict.py', EXPORT / 'predict.py')
print(f'{ONNX.name}: {ONNX.stat().st_size / 1e6:.1f} MB  (best.pt: {best.stat().st_size / 1e6:.1f} MB)')

## 2. Doğrulama: `predict.py` (ONNX) ile PyTorch aynı sonucu veriyor mu?

Test'ten 200 rastgele görüntüde iki yol karşılaştırılır: Ultralytics + PyTorch (`best.pt`, GPU) ve bizim `predict.py` + ONNX Runtime (CPU). Aynı eşik ve NMS ayarlarıyla kutuların neredeyse tamamı eşleşmeli (IoU > 0.9), güven farkı ~0.01'in altında kalmalı. Küçük farklar yeniden boyutlandırma yönteminden gelir.

In [ ]:
def iou1(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / u if u > 0 else 0

sample = test.sample(min(200, len(test)), random_state=0).file.tolist()
det_v = Detector(ONNX, conf={0: 0.05, 1: 0.05}, iou=0.7, providers=['CPUExecutionProvider'])
pt = YOLO(str(best))
n_on = n_pt = n_match = 0; dconf = []; t_onnx = t_pt = 0.0
for f in sample:
    img = cv2.imread(str(DATA / f))
    t0 = time.time(); a = det_v(img); t_onnx += time.time() - t0
    t0 = time.time(); r = pt.predict(img, imgsz=IMGSZ, conf=0.05, iou=0.7, verbose=False)[0]; t_pt += time.time() - t0
    b = list(zip(r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy(), r.boxes.xyxy.cpu().numpy()))
    n_on += len(a); n_pt += len(b)
    for c, s, *bx in a:
        cand = [(iou1(bx, b2), abs(s - s2)) for c2, s2, b2 in b if c2 == c]
        if cand and max(cand)[0] > 0.9:
            n_match += 1; dconf.append(max(cand)[1])
verify = {'görüntü': len(sample), 'kutu (ONNX)': n_on, 'kutu (PyTorch)': n_pt,
          'eşleşen %': round(100 * n_match / max(n_on, 1), 1), 'ort. güven farkı': round(float(np.mean(dconf)) if dconf else -1, 4),
          'ONNX CPU ms/görüntü': round(1000 * t_onnx / len(sample), 1), 'PyTorch GPU ms/görüntü': round(1000 * t_pt / len(sample), 1)}
display(pd.Series(verify))

## 3. Demo: Heinola videoları

Heinola test kareleri videoya ve kare numarasına göre sıralanır. Her video için ayrı bir zaman filtresi çalışır (videolar arası geçişte sıfırlanır).

In [ ]:
hein = test[test.source == 'boreal'].copy()
kv = hein.uid.str.extract(r'^(?P<vid>.+)_frame(?P<fr>\d+)$')
hein['vid'], hein['fr'] = kv.vid, kv.fr.astype(int)
hein = hein.sort_values(['vid', 'fr']).reset_index(drop=True)
vids = hein.groupby('vid').agg(kare=('uid', 'size'), dumanli=('n_smoke', lambda s: int((s > 0).sum()))).sort_values('kare', ascending=False)
display(vids)

det = Detector(ONNX, conf=THR, providers=['CPUExecutionProvider'])
rows = []
t0 = time.time()
for v, g in hein.groupby('vid', sort=False):
    filt = {c: AlarmFilter(K, N) for c in (0, 1)}
    for r in g.itertuples():
        img = cv2.imread(str(DATA / r.file))
        d = det(img)
        raw = {c: any(x[0] == c for x in d) for c in (0, 1)}
        al = {c: filt[c].update(raw[c]) for c in (0, 1)}
        rows.append({'vid': v, 'fr': r.fr, 'uid': r.uid, 'file': r.file, 'gt_smoke': r.n_smoke > 0, 'gt_fire': r.n_fire > 0,
                     'raw_smoke': raw[1], 'raw_fire': raw[0], 'alarm_smoke': al[1], 'alarm_fire': al[0], 'dets': d})
seq = pd.DataFrame(rows)
print(f'{len(seq)} kare · {time.time() - t0:.0f} sn ({1000 * (time.time() - t0) / len(seq):.0f} ms/kare, CPU, okuma dahil)')

## 4. Olay bazında ölçüm (ham kare ↔ filtreli alarm)

- **Yakalama:** dumanlı karelerde tespit/alarm oranı.
- **Yanlış alarm:** dumansız karelerde tespit/alarm oranı.
- **İlk alarm gecikmesi:** her videoda ilk dumanlı kare ile ilk alarm arasındaki kare sayısı. Filtre bekleme yarattığı için ≥ K−1 olması beklenir.

In [ ]:
def rate(mask, col):
    return round(100 * seq.loc[mask, col].mean(), 1) if mask.any() else np.nan

pos, neg = seq.gt_smoke, ~seq.gt_smoke
ev = pd.DataFrame([
    {'ölçüm': 'duman yakalama %', 'ham kare': rate(pos, 'raw_smoke'), 'filtreli alarm': rate(pos, 'alarm_smoke')},
    {'ölçüm': 'duman yanlış alarm %', 'ham kare': rate(neg, 'raw_smoke'), 'filtreli alarm': rate(neg, 'alarm_smoke')},
    {'ölçüm': 'ateş yanlış alarm % (Heinola\'da ateş etiketi yok)', 'ham kare': rate(~seq.gt_fire, 'raw_fire'),
     'filtreli alarm': rate(~seq.gt_fire, 'alarm_fire')}])
display(ev)

delays = []
for v, g in seq.groupby('vid', sort=False):
    g = g.reset_index(drop=True)
    if not g.gt_smoke.any():
        continue
    i0 = int(g.index[g.gt_smoke][0])
    after = g.index[(g.index >= i0) & g.alarm_smoke]
    delays.append({'video': v, 'kare': len(g), 'ilk dumanlı kare': i0,
                   'ilk alarm (kare sonra)': int(after[0] - i0) if len(after) else None,
                   'alarm hiç yok': not len(after)})
delays = pd.DataFrame(delays)
display(delays)
print('videoların', int((~delays['alarm hiç yok']).sum()), '/', len(delays), '\'inde alarm verildi · medyan gecikme',
      delays['ilk alarm (kare sonra)'].median(), 'kare')

## 5. Demo videosu ve GIF

En uzun `MAX_VIDEOS` Heinola videosu art arda. Üst bant alarm durumu (kırmızı = alarm), turuncu kutu ateş, mavi kutu duman tahmini; `SHOW_GT` açıksa ince yeşil çizgi etiket. H.264'e çevrilir ki tarayıcıda ve GitHub'da oynasın.

In [ ]:
pick = vids.index[:MAX_VIDEOS].tolist()
demo = seq[seq.vid.isin(pick)].copy()
demo['vid'] = pd.Categorical(demo.vid, pick, ordered=True)
demo = demo.sort_values(['vid', 'fr'])
raw_mp4, MP4, GIF = Path('/content/demo_raw.mp4'), EXPORT / 'demo_heinola.mp4', ASSETS / 'demo_heinola.gif'
writer = None
for r in demo.itertuples():
    img = cv2.imread(str(DATA / r.file))
    if SHOW_GT:
        lp = DATA / r.file.replace('images/', 'labels/', 1).rsplit('.', 1)[0]
        lp = Path(str(lp) + '.txt')
        H, W = img.shape[:2]
        for ln in (lp.read_text().split('\n') if lp.exists() else []):
            t = ln.split()
            if len(t) == 5:
                _, x, y, w, h = map(float, t)
                cv2.rectangle(img, (int((x - w / 2) * W), int((y - h / 2) * H)), (int((x + w / 2) * W), int((y + h / 2) * H)), (80, 200, 80), 1)
    vis = draw(img, r.dets, {0: r.alarm_fire, 1: r.alarm_smoke}, det.names, info=f'{r.vid} kare {r.fr}')
    s = WIDTH / vis.shape[1]
    vis = cv2.resize(vis, (WIDTH, int(round(vis.shape[0] * s / 2) * 2)))
    cv2.putText(vis, 'Heinola - modelin hic gormedigi yangin', (10, vis.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2, cv2.LINE_AA)
    if writer is None:
        size = (vis.shape[1], vis.shape[0])
        writer = cv2.VideoWriter(str(raw_mp4), cv2.VideoWriter_fourcc(*'mp4v'), FPS, size)
    writer.write(cv2.resize(vis, size) if (vis.shape[1], vis.shape[0]) != size else vis)
writer.release()
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(raw_mp4), '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
                '-movflags', '+faststart', str(MP4)], check=True)
n_gif = min(len(demo), 12 * FPS)
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(raw_mp4), '-frames:v', str(n_gif), '-vf',
                f'fps={FPS},scale=480:-1:flags=lanczos,split[a][b];[a]palettegen=max_colors=96[p];[b][p]paletteuse',
                str(GIF)], check=True)
print(f'{MP4.name}: {len(demo)} kare, {len(demo) / FPS:.0f} sn, {MP4.stat().st_size / 1e6:.1f} MB · {GIF.name}: {GIF.stat().st_size / 1e6:.1f} MB')
from IPython.display import Video
Video(str(MP4), embed=True, width=720)

## 6. README görselleri ve özet

In [ ]:
log = pd.read_csv(REP / 'experiments.csv').sort_values('EXP')
lab = log.EXP.str.split('_').str[0] + '\n' + log['not'].str.slice(0, 22)
x = np.arange(len(log)); w = 0.38
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - w / 2, log['mAP50_fire'], w, label='fire mAP50', color='#e8590c')
ax.bar(x + w / 2, log['mAP50_smoke'], w, label='smoke mAP50', color='#4dabf7')
for i, (f, s) in enumerate(zip(log['mAP50_fire'], log['mAP50_smoke'])):
    ax.text(i - w / 2, f + 0.01, f'{f:.2f}', ha='center', fontsize=8); ax.text(i + w / 2, s + 0.01, f'{s:.2f}', ha='center', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(lab, fontsize=7); ax.set_ylim(0, 1); ax.set_ylabel('val mAP50')
ax.set_title('Deneyler (val) — her adım bir değişiklik'); ax.legend(); plt.tight_layout()
plt.savefig(ASSETS / 'experiments.png', dpi=120); plt.show()

for f in ['val_threshold.png', 'test_miss_fire.png', 'test_false_alarm.png', 'test_miss_heinola.png']:
    if (REP / 'final_eval' / f).exists():
        shutil.copy(REP / 'final_eval' / f, ASSETS / f)
for f in ['final_eval.md', 'dataset_v2.md', 'dataset_v3.md', 'experiments.csv', 'experiments_diag.csv']:
    if (REP / f).exists():
        shutil.copy(REP / f, EXPORT / f)

L = ['# Demo ve export özeti', '',
     f'- ONNX: `{ONNX.name}` ({ONNX.stat().st_size / 1e6:.1f} MB), girdi {IMGSZ}×{IMGSZ}',
     '- Doğrulama (ONNX + predict.py ↔ PyTorch): ' + ' · '.join(f'{k} {v}' for k, v in verify.items()),
     f'- Zaman filtresi: son {N} karenin en az {K}\'sında tespit → alarm · eşikler fire {THR[0]}, smoke {THR[1]}', '',
     '## Heinola (görülmemiş yangın), olay bazında', '', ev.to_markdown(index=False), '', delays.to_markdown(index=False), '',
     f'- Demo: `demo_heinola.mp4` ({len(demo)} kare, {len(pick)} video) · `assets/demo_heinola.gif`']
txt = '\n'.join(L)
(REP / 'demo_summary.md').write_text(txt); shutil.copy(REP / 'demo_summary.md', EXPORT / 'demo_summary.md')
print(txt)
print('\nexport klasörü:', sorted(p.name for p in EXPORT.iterdir()))

**Bana gönderilecekler:** son hücrenin çıktısı ve demo videosu hakkında bir iki cümle (alarm bandı mantıklı zamanlarda mı yanıyor?). README'yi bu sayılarla tamamlayacağım.